In [ ]:
import os
import warnings

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["POLARS_MAX_THREADS"] = "1"

%matplotlib inline

# Keep sklearn's parallel warning propagation initialized.
if not warnings.filters:
    warnings.simplefilter("default")

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats
from patsy import dmatrix
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").exists()),
    Path.cwd(),
)
OUTPUT_DIR = Path(os.environ.get(
    "SI_SHAP_NOTEBOOK_OUTPUT_DIR",
    PROJECT_ROOT / "outputs" / "unadjusted_vs_random",
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT_DIR / "unadjusted_vs_random_results.csv"

In [ ]:
def spline_pvalues(x, y, sigma):
    """Return unknown- and known-variance p-values from one projection."""
    design = np.asarray(dmatrix(
        "bs(x, df=3, degree=3, include_intercept=False) - 1",
        {"x": np.asarray(x)},
    ))
    centered_design = design - design.mean(axis=0, keepdims=True)
    left, singular_values, _ = np.linalg.svd(centered_design, full_matrices=False)
    tolerance = max(centered_design.shape) * np.finfo(float).eps * singular_values[0]
    rank = int(np.sum(singular_values > tolerance))
    basis = left[:, :rank]
    centered_y = np.asarray(y) - np.mean(y)
    coordinates = basis.T @ centered_y
    model_ss = float(coordinates @ coordinates)
    residual = centered_y - basis @ coordinates
    residual_df = y.size - rank - 1
    f_statistic = (model_ss / rank) / (float(residual @ residual) / residual_df)
    return {
        "unknown": float(stats.f.sf(f_statistic, rank, residual_df)),
        "known": float(stats.chi2.sf(model_ss / sigma**2, df=rank)),
    }


def run_simulation(
    n_iters, n_samples, n_features, k_select, alpha=0.05, seed=123,
    sigma=1.0, rf_params=None, verbose=True,
    show_progress=False, progress_desc=None,
):
    if n_iters < 2 or n_samples <= 4 or n_features < 1:
        raise ValueError("Use n_iters >= 2, n_samples > 4, and n_features >= 1.")
    k_values = np.atleast_1d(k_select).tolist()
    if any(not isinstance(k, (int, np.integer)) for k in k_values):
        raise ValueError("Every k_select value must be an integer.")
    k_values = sorted(set(int(k) for k in k_values))
    if not k_values or any(not 1 <= k <= n_features for k in k_values) or not 0 < alpha < 1:
        raise ValueError("Require 1 <= k_select <= n_features and 0 < alpha < 1.")
    max_k = max(k_values)
    if not np.isscalar(sigma) or not np.isfinite(sigma) or sigma <= 0:
        raise ValueError("sigma must be finite and positive.")
    # Keep the forest configuration in one adjustable dictionary.
    forest_params = {"n_estimators": 50, "max_depth": 5, "random_state": 42}
    if rf_params is not None:
        forest_params.update(dict(rf_params))
    if forest_params.get("random_state") is None:
        raise ValueError("random_state must be fixed for reproducibility.")
    data_seed, random_seed = np.random.SeedSequence(seed).spawn(2)
    data_rng = np.random.default_rng(data_seed)
    random_rng = np.random.default_rng(random_seed)
    pvals_shap = {"unknown": [], "known": []}
    pvals_random = {"unknown": [], "known": []}
    
    iterations = tqdm(
        range(n_iters), desc=progress_desc or "known + unknown variance",
        unit="iteration", leave=True, disable=not show_progress,
    )
    for _ in iterations:
        # [1] Generate data under the global null hypothesis.
        X = data_rng.standard_normal((n_samples, n_features))
        y = sigma * data_rng.standard_normal(n_samples)
        
        # [2] Fit a random forest and compute SHAP importance scores.
        model = RandomForestRegressor(**forest_params)
        model.fit(X, y)
        
        explainer = shap.TreeExplainer(
            model, feature_perturbation="tree_path_dependent"
        )
        shap_values = np.asarray(explainer.shap_values(X))
        if shap_values.shape != X.shape or not np.all(np.isfinite(shap_values)):
            raise ValueError(f"Unexpected SHAP output shape {shap_values.shape}")
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        
        # [3] Select features using SHAP importance and random sampling.
        # Random selection
        random_selected_idx = random_rng.permutation(n_features)[:max_k]
        # SHAP-based selection
        shap_selected_idx = np.lexsort((np.arange(n_features), -mean_abs_shap))[:max_k]
        
        # [4] Compute each needed B-spline projection once.
        tested_features = set(random_selected_idx) | set(shap_selected_idx)
        pvalues_by_feature = {
            int(idx): spline_pvalues(X[:, idx], y, sigma)
            for idx in tested_features
        }
        # Run the B-spline test after random selection.
        for idx in random_selected_idx:
            values = pvalues_by_feature[int(idx)]
            for variance, p_value in values.items():
                pvals_random[variance].append(p_value)
        
        # [5] Run the unadjusted test after SHAP-based selection.
        for idx in shap_selected_idx:
            values = pvalues_by_feature[int(idx)]
            for variance, p_value in values.items():
                pvals_shap[variance].append(p_value)

    # [6] Calculate the empirical false-positive rate (FPR).
    pvals_shap = {name: np.asarray(values).reshape(n_iters, max_k)
                   for name, values in pvals_shap.items()}
    pvals_random = {name: np.asarray(values).reshape(n_iters, max_k)
                     for name, values in pvals_random.items()}
    rows = []
    for variance in ("unknown", "known"):
        shap_rejections = pvals_shap[variance] < alpha
        random_rejections = pvals_random[variance] < alpha
        for k in k_values:
            shap_rates = shap_rejections[:, :k].mean(axis=1)
            random_rates = random_rejections[:, :k].mean(axis=1)
            rows.extend([
                {"variance": variance, "k_select": k, "method": "Random baseline",
                 "fpr": random_rates.mean(), "se": random_rates.std(ddof=1) / np.sqrt(n_iters)},
                {"variance": variance, "k_select": k, "method": "Unadjusted post-SHAP",
                 "fpr": shap_rates.mean(), "se": shap_rates.std(ddof=1) / np.sqrt(n_iters)},
            ])
    summary = pd.DataFrame(rows)
    if verbose:
        print("\n=== Simulation Results: known and unknown variance ===")
        display(summary)
    
    # Store one row per method and selection size.
    return {"sigma": sigma, "k_values": k_values,
            "rf_params": forest_params.copy(),
            "pvals_shap": pvals_shap,
            "pvals_random": pvals_random,
            "summary": summary}

In [ ]:
K_SELECT_VALUES = list(range(1, 11))
N_ITERS = int(os.environ.get("SI_SHAP_NOTEBOOK_N_ITERS", "1000"))
# Set SI_SHAP_NOTEBOOK_N_ITERS=2 for a fast end-to-end execution check.

# Add or edit named configurations to test other RandomForestRegressor settings.
RF_CONFIGS = {
    "baseline": {
        "n_estimators": 50, "max_depth": 5,
        "min_samples_leaf": 1, "max_features": 1.0,
    },
    "larger_forest": {
        "n_estimators": 200, "max_depth": None,
        "min_samples_leaf": 1, "max_features": 1.0,
    },
    "regularized_forest": {
        "n_estimators": 100, "max_depth": 3,
        "min_samples_leaf": 5, "max_features": 0.7,
    },
}

# These shared options make every configuration reproducible and parallelize fitting.
RF_SHARED_PARAMS = {"random_state": 42, "n_jobs": 32}

results = {}
for rf_name, config in RF_CONFIGS.items():
    rf_params = {**RF_SHARED_PARAMS, **config}
    result = run_simulation(
        n_iters=N_ITERS, n_samples=100, n_features=20,
        k_select=K_SELECT_VALUES, seed=123, sigma=1.0,
        rf_params=rf_params, verbose=False, show_progress=True,
        progress_desc=f"{rf_name} | known + unknown",
    )
    result["summary"] = result["summary"].assign(rf_config=rf_name)
    results[rf_name] = result

combined_results = pd.concat(
    [result["summary"] for result in results.values()], ignore_index=True
)
combined_results.to_csv(RESULTS_CSV, index=False)
print(f"Saved simulation results to {RESULTS_CSV}")
display(combined_results)

In [ ]:
# Reload the saved results so this analysis does not depend on simulation state.
analysis_results = pd.read_csv(RESULTS_CSV)

# Quantify how much each configuration differs from the baseline forest.
FPR_TOLERANCE = 0.02  # Define 'similar' as no FPR difference larger than 2 points.
comparison_keys = ["variance", "k_select", "method"]
baseline_fpr = (
    analysis_results.loc[analysis_results["rf_config"] == "baseline", comparison_keys + ["fpr"]]
    .rename(columns={"fpr": "baseline_fpr"})
)
comparison = analysis_results.merge(baseline_fpr, on=comparison_keys, how="left")
comparison["fpr_difference"] = comparison["fpr"] - comparison["baseline_fpr"]
comparison["absolute_difference"] = comparison["fpr_difference"].abs()

similarity_summary = (
    comparison.groupby(["rf_config", "variance", "method"], as_index=False)
    .agg(mean_absolute_difference=("absolute_difference", "mean"),
         max_absolute_difference=("absolute_difference", "max"))
)
similarity_summary["similar_to_baseline"] = (
    similarity_summary["max_absolute_difference"] <= FPR_TOLERANCE
)
fpr_pivot = comparison.pivot(
    index="k_select", columns=["variance", "method", "rf_config"], values="fpr"
)
comparison.to_csv(
    OUTPUT_DIR / "unadjusted_vs_random_comparison.csv", index=False
)
similarity_summary.to_csv(
    OUTPUT_DIR / "unadjusted_vs_random_similarity_summary.csv", index=False
)
fpr_pivot.to_csv(OUTPUT_DIR / "unadjusted_vs_random_fpr_pivot.csv")
display(similarity_summary)
display(fpr_pivot)

In [ ]:
# Plot only from the saved CSV; rerunning the simulation is not required.
plot_results = pd.read_csv(RESULTS_CSV)
required_columns = {"variance", "k_select", "method", "fpr", "se", "rf_config"}
missing_columns = required_columns.difference(plot_results.columns)
if missing_columns:
    raise ValueError(f"Results CSV is missing columns: {sorted(missing_columns)}")
plot_k_values = sorted(plot_results["k_select"].unique())

# The random baseline is identical across forest settings, so plot it once per panel.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, variance in zip(axes, ["unknown", "known"]):
    panel = plot_results[plot_results["variance"] == variance]
    random_line = panel[(panel["method"] == "Random baseline") &
                        (panel["rf_config"] == "baseline")].sort_values("k_select")
    ax.plot(random_line["k_select"], random_line["fpr"],
            color="gray", linestyle="--", marker="o", label="Random baseline")
    shap_panel = panel[panel["method"] == "Unadjusted post-SHAP"]
    for rf_name, group in shap_panel.groupby("rf_config", sort=False):
        group = group.sort_values("k_select")
        ax.errorbar(group["k_select"], group["fpr"], yerr=group["se"],
                    marker="o", capsize=2, label=f"SHAP: {rf_name}")
    ax.axhline(0.05, color="red", linestyle=":", linewidth=1.5, label="alpha = 0.05")
    ax.set(title=f"{variance.capitalize()} variance", xlabel="k_select",
           xticks=plot_k_values)
    ax.set_ylim(bottom=0)
    ax.grid(True, linestyle=":", alpha=0.6)
axes[0].set_ylabel("Selected-hypothesis FPR")
axes[1].legend(fontsize=8)
fig.suptitle("Sensitivity of post-SHAP results to random-forest parameters")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "unadjusted_vs_random_rf_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()